# Sentinel — EDA (Phase 1, Step 3)

Exploration of **BAF `Base.csv`** (Bank Account Fraud, NeurIPS 2022). Read-only; conclusions feed preprocessing (Step 4).

Run from the repo root: `jupyter lab notebooks/eda.ipynb`. The cells re-derive every number in `reports/eda_findings.md`.

In [ ]:
import sys, pathlib
# Make the repo root importable so `ml.src` resolves when running from notebooks/.
ROOT = pathlib.Path.cwd()
if (ROOT / 'ml').exists() is False and (ROOT.parent / 'ml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
from ml.src import config

df = pd.read_csv(config.DATA_RAW / 'Base.csv')
T = config.TARGET
df.shape

## Dtypes & feature types
Five true categoricals (all low cardinality), the rest numeric.

In [ ]:
print(df.dtypes.value_counts())
obj = df.select_dtypes('object').columns.tolist()
print('categorical:', obj)

## Class imbalance
~1.10% fraud (≈89:1). This is why we optimize **PR-AUC / recall**, not accuracy: predicting “never fraud” scores 98.9% accuracy and catches zero fraud.

In [ ]:
print(df[T].value_counts())
print('fraud rate: %.4f%%' % (100 * df[T].mean()))

## Temporal structure (`month` 0–7)
Row counts decline over time and fraud rate **drifts upward** in later months (concept drift). This motivates a temporal split and feeds the Step 8 cost analysis (optimal threshold shifts with base rate).

In [ ]:
g = df.groupby('month')[T].agg(['count', 'sum', 'mean'])
g['fraud_rate_%'] = (100 * g['mean']).round(3)
g[['count', 'sum', 'fraud_rate_%']]

In [ ]:
# Visual: fraud rate per month
import matplotlib.pyplot as plt
ax = g['fraud_rate_%'].plot(marker='o', figsize=(7, 3), title='Fraud rate by month (%)')
ax.set_xlabel('month'); ax.set_ylabel('fraud rate %'); plt.tight_layout()

## Cardinality
Categoricals are all 2–7 distinct values → one-hot is cheap; no high-cardinality blow-up. Several numerics are pre-bucketed (`customer_age`, `income`).

In [ ]:
for c in obj:
    print(f'{c:22s} nunique={df[c].nunique():>3}  {list(df[c].unique())}')
print()
for c in df.columns:
    if c != T and df[c].dtype != 'object' and df[c].nunique() <= 12:
        print(f'{c:32s} nunique={df[c].nunique():>3}  {sorted(df[c].unique())[:12]}')

## Missingness via negative sentinels
No explicit NaNs. BAF encodes “unknown” as negatives. Distinguish three groups:
- **`-1` = missing** count fields → add missing flag, map -1→NaN.
- **`intended_balcon_amount`**: continuous negatives (74%) → own missing flag.
- **Genuinely-negative** (`credit_risk_score`, `velocity_6h`) → keep as real values.

In [ ]:
print('explicit NaN:', df.isna().sum().sum())
num = df.select_dtypes(include=[np.number]).columns
for c in num:
    neg = int((df[c] < 0).sum())
    if neg:
        print(f'{c:30s} negatives={neg:>8} ({100*neg/len(df):5.1f}%) min={df[c].min()}')

## Leakage & constant columns
Max |corr(feature, target)| ≈ 0.07 → no single feature leaks the label. `device_fraud_count` is constant (all 0) → drop it.

In [ ]:
corr = df[num].corr()[T].drop(T).abs().sort_values(ascending=False)
print(corr.head(8).round(3))
print()
print('constant columns:', [c for c in df.columns if df[c].nunique() <= 1])